# FX-AGENT — ₹0 Colab Runner

**No Kie.ai API, no GitHub token, no broker account.** Runs the public FX-AGENT repository in Colab using standard-library Python.

This starter checks source code and runs a **synthetic demonstration**, not a live Forex trading bot or a profitability test. Do not place any API keys in this notebook, including its saved outputs.

**First:** make sure your public GitHub repository has `fxagent/`, `tests/`, and `README.md` at its root. Otherwise the verification cell will stop rather than claim success.

In [ ]:
# CELL 1 — Read-only clone from your public GitHub repository.
from pathlib import Path
from urllib.parse import urlparse
import subprocess, sys, tempfile, re

repo_url = input('Paste your PUBLIC FX-AGENT GitHub repository URL: ').strip().rstrip('/')
parsed = urlparse(repo_url)
parts = parsed.path.strip('/').removesuffix('.git').split('/')
if (parsed.scheme != 'https' or parsed.netloc.lower() != 'github.com'
        or len(parts) != 2 or not all(re.fullmatch(r'[A-Za-z0-9_.-]+', x) for x in parts)):
    raise ValueError('Use the exact format https://github.com/YOUR_USERNAME/FX-AGENT')
if parts[1] != 'FX-AGENT':
    raise ValueError('Expected repository name FX-AGENT; check your URL')
repo_url = f'https://github.com/{parts[0]}/{parts[1]}'

project = Path(tempfile.mkdtemp(prefix='fx-agent-public-')) / 'FX-AGENT'
subprocess.run(['git', 'clone', '--depth', '1', repo_url + '.git', str(project)], check=True, timeout=90)
if not (project / 'fxagent' / 'backtest.py').is_file() or not (project / 'tests' / 'test_foundation.py').is_file():
    raise RuntimeError('Repo cloned, but FX-AGENT project files were not uploaded to the repository root. Upload the extracted project folders first.')
print('Public repository downloaded:', project)

In [ ]:
# CELL 2 — Run all tests. This cell stops on failure.
import subprocess, sys
completed = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=project, check=True, text=True, capture_output=True, timeout=90
)
print(completed.stdout)
print(completed.stderr)
print('PASS — all unit tests completed successfully.')

In [ ]:
# CELL 3 — Run the synthetic engineering demonstration (NOT real market data).
completed = subprocess.run(
    [sys.executable, '-m', 'fxagent.demo'],
    cwd=project, check=True, text=True, capture_output=True, timeout=90
)
print(completed.stdout)
print(completed.stderr)

In [ ]:
# CELL 4 — Read one generated report (do not treat this as evidence of profits).
import json
reports = sorted((project / 'reports').glob('*.json'), key=lambda path: path.stat().st_mtime)
if not reports:
    raise FileNotFoundError('Run CELL 3 first to generate a report.')
latest_report = reports[-1]
record = json.loads(latest_report.read_text(encoding='utf-8'))
r = record['result']
print('Report file:', latest_report.name)
print('Symbol:', r['symbol'])
print('Partition:', r['partition'])
print('Synthetic trades:', r['number_of_trades'])
print('Final cash:', r['final_cash'], r['account_currency'])
print('Net P&L:', r['net_pnl'], r['account_currency'])
print('Synthetic example only: NOT a profit estimate.')

In [ ]:
# CELL 5 — Optional: download only this synthetic report to your Chromebook.
# Run only if you actually want the JSON file; no internet API call is made.
from google.colab import files
files.download(str(latest_report))

## Updating your code

Edit Python source files on GitHub, commit them, then rerun Cell 1 to clone the latest public version. Since Colab uses a temporary runtime, always keep source files on GitHub.

GitHub Actions also runs offline unit tests on each push if you included `.github/workflows/tests.yml`. View status under **GitHub → Actions**. Do not commit generated `data/` or `reports/` folders, saved secrets, or personally identifying information.